# GMMs — Gaussian Mixture Models
**Mathematics for Machine Learning, Ch 11**

This notebook implements GMM density estimation and the EM algorithm from scratch.
Topics covered:
1. Gaussian mixture density (1D, 3 components)
2. EM algorithm from scratch (E-step + M-step)
3. Log-likelihood convergence
4. K-means vs GMM comparison
5. Model selection via BIC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from scipy.stats import multivariate_normal
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

---
## 1. Gaussian Mixture Density (1D)

A GMM with $K$ components is a weighted sum of Gaussians:
$$p(x) = \sum_{k=1}^K \pi_k \, \mathcal{N}(x \mid \mu_k, \sigma_k^2)$$
subject to $\pi_k \geq 0$, $\sum_k \pi_k = 1$.

Below we plot a 1D GMM with 3 components, showing each component individually and the mixture density.

In [ ]:
# 1D GMM parameters
mus_1d    = np.array([-3.0, 0.5, 4.0])
sigs_1d   = np.array([0.8, 1.2, 0.6])
pis_1d    = np.array([0.3, 0.4, 0.3])
K_1d      = 3

x_grid = np.linspace(-7, 7, 500)

# Individual component densities
components_1d = np.array([
    pis_1d[k] * multivariate_normal.pdf(x_grid, mean=mus_1d[k], cov=sigs_1d[k]**2)
    for k in range(K_1d)
])  # (K, 500)

mixture_1d = components_1d.sum(axis=0)

colors = ['steelblue', 'tomato', 'seagreen']

fig, ax = plt.subplots(figsize=(10, 4))
for k in range(K_1d):
    ax.plot(x_grid, components_1d[k], '--', color=colors[k], lw=1.8,
            label=f'$\\pi_{k+1}\\mathcal{{N}}(\\mu={mus_1d[k]},\\,\\sigma={sigs_1d[k]})$ weight={pis_1d[k]}')
    ax.fill_between(x_grid, components_1d[k], alpha=0.12, color=colors[k])

ax.plot(x_grid, mixture_1d, 'k-', lw=2.5, label='GMM mixture $p(x)$')
ax.set_xlabel('x')
ax.set_ylabel('Density')
ax.set_title('1D Gaussian Mixture Model — 3 components')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'Integral of mixture (approx): {np.trapz(mixture_1d, x_grid):.6f}  (should be ≈ 1.0)')

---
## 2. EM Algorithm From Scratch

### The EM equations for GMMs

**E-step** — compute responsibilities (posterior probability that component $k$ generated point $n$):
$$r_{nk} = \frac{\pi_k \mathcal{N}(x_n \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \pi_j \mathcal{N}(x_n \mid \mu_j, \Sigma_j)}$$

**M-step** — update parameters using weighted statistics:
$$N_k = \sum_n r_{nk} \qquad \mu_k \leftarrow \frac{1}{N_k}\sum_n r_{nk} x_n$$
$$\Sigma_k \leftarrow \frac{1}{N_k}\sum_n r_{nk}(x_n - \mu_k)(x_n-\mu_k)^\top \qquad \pi_k \leftarrow \frac{N_k}{N}$$

In [ ]:
# Generate 2D mixture data
N_data = 300
K_true = 3

true_mus   = np.array([[-3, -2], [2, 2], [0, -3]], dtype=float)
true_Sigmas= [np.array([[1.0, 0.5], [0.5, 0.8]]),
              np.array([[0.6, -0.3], [-0.3, 1.2]]),
              np.array([[1.5, 0.0], [0.0, 0.4]])]
true_pis   = np.array([0.3, 0.4, 0.3])

# Sample from the true GMM
samples = []
labels_true = []
for k, (mu, Sig, pi) in enumerate(zip(true_mus, true_Sigmas, true_pis)):
    n_k = int(N_data * pi)
    samples.append(np.random.multivariate_normal(mu, Sig, size=n_k))
    labels_true.extend([k] * n_k)
X_2d = np.vstack(samples)  # (N, 2)
labels_true = np.array(labels_true)

print(f'Generated {len(X_2d)} data points from a {K_true}-component GMM in 2D')

In [ ]:
def gaussian_density(X, mu, Sigma):
    """Multivariate Gaussian density for each row of X."""
    return multivariate_normal.pdf(X, mean=mu, cov=Sigma, allow_singular=True)


def e_step(X, mus, Sigmas, pis):
    """Compute responsibilities r[n,k] for all data points and components."""
    N, K = X.shape[0], len(mus)
    R = np.zeros((N, K))
    for k in range(K):
        R[:, k] = pis[k] * gaussian_density(X, mus[k], Sigmas[k])
    # Normalize (avoid division by zero)
    row_sums = R.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1e-300, row_sums)
    return R / row_sums  # (N, K)


def m_step(X, R):
    """Update GMM parameters given responsibilities."""
    N, K = R.shape
    D = X.shape[1]
    Nk   = R.sum(axis=0)             # (K,) effective counts
    pis  = Nk / N                    # (K,) updated weights
    mus  = np.zeros((K, D))
    Sigmas = []
    for k in range(K):
        # Weighted mean
        mus[k] = (R[:, k, None] * X).sum(axis=0) / Nk[k]
        # Weighted covariance
        diff   = X - mus[k]          # (N, D)
        Sigma_k = (R[:, k, None] * diff).T @ diff / Nk[k]
        Sigma_k += 1e-6 * np.eye(D)  # regularize to prevent degeneracy
        Sigmas.append(Sigma_k)
    return mus, Sigmas, pis


def log_likelihood(X, mus, Sigmas, pis):
    """Compute GMM log-likelihood: Σ_n log Σ_k π_k N(x_n | μ_k, Σ_k)."""
    N, K = X.shape[0], len(mus)
    joint = np.zeros((N, K))
    for k in range(K):
        joint[:, k] = pis[k] * gaussian_density(X, mus[k], Sigmas[k])
    return np.log(joint.sum(axis=1) + 1e-300).sum()


def run_em(X, K, n_iter=20, seed=0):
    """Run EM algorithm for a K-component GMM. Returns history of parameters and log-likelihoods."""
    rng = np.random.RandomState(seed)
    D   = X.shape[1]
    # Initialize from random data points
    idx  = rng.choice(len(X), K, replace=False)
    mus  = X[idx].copy()
    Sigmas = [np.eye(D) for _ in range(K)]
    pis  = np.ones(K) / K

    history = {'mus': [mus.copy()], 'Sigmas': [Sigmas.copy()],
               'pis': [pis.copy()], 'log_lik': []}

    for _ in range(n_iter):
        R    = e_step(X, mus, Sigmas, pis)
        mus, Sigmas, pis = m_step(X, R)
        ll   = log_likelihood(X, mus, Sigmas, pis)
        history['mus'].append(mus.copy())
        history['Sigmas'].append(Sigmas.copy())
        history['pis'].append(pis.copy())
        history['log_lik'].append(ll)

    return mus, Sigmas, pis, history


# Run EM for 20 iterations
K_fit = 3
mus_fit, Sigmas_fit, pis_fit, history = run_em(X_2d, K=K_fit, n_iter=20, seed=1)
print('EM completed.')
print(f'Final log-likelihood: {history["log_lik"][-1]:.2f}')
print(f'Final weights π: {pis_fit.round(3)}')
print(f'Final means μ:')
for k in range(K_fit):
    print(f'  μ_{k+1} = {mus_fit[k].round(3)}')

In [ ]:
def draw_ellipse(ax, mu, Sigma, color, alpha=0.25, n_std=2.0):
    """Draw a 2-sigma confidence ellipse for a 2D Gaussian."""
    vals, vecs = np.linalg.eigh(Sigma)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h  = 2 * n_std * np.sqrt(vals)
    ell   = Ellipse(xy=mu, width=w, height=h, angle=angle,
                    facecolor=color, alpha=alpha, edgecolor=color, lw=2)
    ax.add_patch(ell)


def plot_gmm_state(ax, X, mus, Sigmas, pis, title, iter_label=None):
    """Plot data points colored by hard assignment from responsibilities, with ellipses."""
    R = e_step(X, mus, Sigmas, pis)
    assignments = R.argmax(axis=1)
    cluster_colors = ['steelblue', 'tomato', 'seagreen']
    for k in range(len(mus)):
        mask = assignments == k
        ax.scatter(X[mask, 0], X[mask, 1], alpha=0.4, s=15,
                   color=cluster_colors[k], label=f'Cluster {k+1}')
        draw_ellipse(ax, mus[k], Sigmas[k], color=cluster_colors[k])
        ax.scatter(*mus[k], s=100, color=cluster_colors[k], edgecolors='black',
                   zorder=5, marker='*')
    ax.set_title(title)
    if iter_label:
        ax.text(0.02, 0.97, iter_label, transform=ax.transAxes,
                va='top', fontsize=9, color='black',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))


# Static convergence plot: initial / iter 5 / final
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
snapshots = [
    (0,  'Initial state (iter 0)'),
    (5,  'After 5 iterations'),
    (20, 'After 20 iterations (final)'),
]

for ax, (it, title) in zip(axes, snapshots):
    plot_gmm_state(
        ax, X_2d,
        history['mus'][it],
        history['Sigmas'][it],
        history['pis'][it],
        title,
        iter_label=f'LL={history["log_lik"][it-1]:.1f}' if it > 0 else 'initializing'
    )

plt.suptitle('EM Algorithm Convergence (2D GMM, K=3)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Log-Likelihood Convergence

The EM guarantee: the log-likelihood $\ell(\theta)$ is **non-decreasing** at every iteration.
EM converges to a local maximum (not guaranteed to be global).

In [ ]:
# Run EM with multiple restarts to show log-likelihood curves
n_restarts = 4
n_iter_conv = 30

fig, ax = plt.subplots(figsize=(9, 4))
colors_restart = ['steelblue', 'tomato', 'seagreen', 'purple']

best_ll = -np.inf
for restart in range(n_restarts):
    _, _, _, h = run_em(X_2d, K=K_fit, n_iter=n_iter_conv, seed=restart * 7)
    lls = h['log_lik']
    ax.plot(range(1, len(lls) + 1), lls, '-o', markersize=3,
            color=colors_restart[restart], lw=1.8,
            label=f'Restart {restart+1} (final LL={lls[-1]:.1f})')
    if lls[-1] > best_ll:
        best_ll = lls[-1]

ax.set_xlabel('EM Iteration')
ax.set_ylabel('Log-likelihood $\\ell(\\theta)$')
ax.set_title('Log-likelihood vs EM iteration (4 random restarts, K=3)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Best final log-likelihood across restarts: {best_ll:.2f}')
print('Note: EM monotonically increases log-likelihood within each restart.')
print('Different restarts find different local optima.')

---
## 4. K-means vs GMM

K-means is the hard-assignment limit of EM for GMMs (with equal spherical covariances $\Sigma_k = \sigma^2 I$ as $\sigma^2 \to 0$).

| Property | K-means | EM for GMMs |
|----------|---------|-------------|
| Assignment | Hard (0 or 1) | Soft ($r_{nk} \in [0,1]$) |
| Cluster shape | Spherical | Arbitrary ellipsoid |
| Covariance | Implicit $\sigma^2 I$ | Learned per cluster |
| Objective | Sum of squared distances | Log-likelihood |

In [ ]:
# Fit K-means
km = KMeans(n_clusters=K_fit, random_state=42, n_init=10)
km.fit(X_2d)
labels_km = km.labels_

# Final GMM assignments from scratch EM
R_final = e_step(X_2d, mus_fit, Sigmas_fit, pis_fit)
labels_gmm = R_final.argmax(axis=1)

# sklearn GMM for clean comparison plot
gmm_sk = GaussianMixture(n_components=K_fit, random_state=42, n_init=5)
gmm_sk.fit(X_2d)
labels_gmm_sk = gmm_sk.predict(X_2d)

# Decision region meshgrid
x1_range = np.linspace(X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1, 200)
x2_range = np.linspace(X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1, 200)
xx, yy   = np.meshgrid(x1_range, x2_range)
grid     = np.c_[xx.ravel(), yy.ravel()]

# K-means regions
km_grid = km.predict(grid).reshape(xx.shape)
# GMM regions
gmm_grid = gmm_sk.predict(grid).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cmap_bg = plt.cm.Pastel1

# K-means plot
ax = axes[0]
ax.contourf(xx, yy, km_grid, alpha=0.25, cmap=cmap_bg)
scatter_colors = ['steelblue', 'tomato', 'seagreen']
for k in range(K_fit):
    mask = labels_km == k
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], alpha=0.5, s=18,
               color=scatter_colors[k])
ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
           s=200, marker='X', color='black', zorder=5, label='centroids')
ax.set_title('K-means (hard assignment)')
ax.legend()

# GMM plot
ax = axes[1]
ax.contourf(xx, yy, gmm_grid, alpha=0.25, cmap=cmap_bg)
for k in range(K_fit):
    mask = labels_gmm_sk == k
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], alpha=0.5, s=18,
               color=scatter_colors[k])
    draw_ellipse(ax, gmm_sk.means_[k], gmm_sk.covariances_[k],
                 color=scatter_colors[k])
ax.scatter(gmm_sk.means_[:, 0], gmm_sk.means_[:, 1],
           s=200, marker='*', color='black', zorder=5, label='means')
ax.set_title('GMM via EM (soft assignment, elliptical clusters)')
ax.legend()

plt.suptitle('K-means vs GMM: Same Data, Different Assumptions', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Model Selection via BIC

The **Bayesian Information Criterion** penalizes model complexity:
$$\text{BIC}(K) = -2\,\ell(\hat{\theta}) + p(K)\log N$$
where $p(K)$ is the number of free parameters (means + covariances + weights).

Choose $K^* = \arg\min_K \text{BIC}(K)$.

In [ ]:
K_values = range(1, 7)
bics     = []
lls      = []

for K in K_values:
    # Use sklearn for reliable convergence across K values
    gmm = GaussianMixture(n_components=K, random_state=42, n_init=5, max_iter=200)
    gmm.fit(X_2d)
    bics.append(gmm.bic(X_2d))
    lls.append(gmm.score(X_2d) * len(X_2d))  # total log-likelihood

best_K = list(K_values)[np.argmin(bics)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Log-likelihood
ax = axes[0]
ax.plot(K_values, lls, 's-', color='steelblue', lw=2, markersize=7)
ax.set_xlabel('Number of components K')
ax.set_ylabel('Log-likelihood $\\ell(\\hat{\\theta})$')
ax.set_title('Log-likelihood vs K\n(always increases — no penalization)')
ax.set_xticks(list(K_values))
ax.grid(True, alpha=0.3)

# BIC
ax = axes[1]
ax.plot(K_values, bics, 'o-', color='tomato', lw=2, markersize=7)
ax.axvline(best_K, color='navy', ls='--', lw=1.5, label=f'Best K={best_K}')
ax.scatter([best_K], [bics[best_K - 1]], s=150, color='navy', zorder=5)
ax.set_xlabel('Number of components K')
ax.set_ylabel('BIC')
ax.set_title('BIC vs K\n(minimum = best model)')
ax.set_xticks(list(K_values))
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('GMM Model Selection via BIC', fontweight='bold')
plt.tight_layout()
plt.show()

print('BIC values by K:')
for K, bic in zip(K_values, bics):
    marker = '  <-- minimum (best K)' if K == best_K else ''
    print(f'  K={K}: BIC={bic:.1f}{marker}')

print(f'\nTrue K used to generate data: {K_true}')
print(f'BIC-selected K: {best_K}')

---
## Summary

| Concept | Formula |
|---------|----------|
| Mixture density | $p(x) = \sum_k \pi_k \mathcal{N}(x \mid \mu_k, \Sigma_k)$ |
| Responsibility (E-step) | $r_{nk} = \frac{\pi_k \mathcal{N}(x_n \mid \mu_k, \Sigma_k)}{\sum_j \pi_j \mathcal{N}(x_n \mid \mu_j, \Sigma_j)}$ |
| Effective count | $N_k = \sum_n r_{nk}$ |
| Mean update (M-step) | $\mu_k \leftarrow \frac{1}{N_k}\sum_n r_{nk} x_n$ |
| Covariance update (M-step) | $\Sigma_k \leftarrow \frac{1}{N_k}\sum_n r_{nk}(x_n-\mu_k)(x_n-\mu_k)^\top$ |
| Weight update (M-step) | $\pi_k \leftarrow N_k / N$ |
| BIC | $\text{BIC}(K) = -2\ell(\hat{\theta}) + p(K)\log N$ |